In [1]:
!pip install transformers datasets scikit-learn sentencepiece accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 141.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [10]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

print("Loading datasets...")
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/NLPLabs-2024/Dont_Patronize_Me_Trainingset/'

#load data
df_raw = pd.read_csv(base_path + 'dontpatronizeme_pcl.tsv', sep='\t', skiprows=4, header=None, on_bad_lines='skip')
df_raw.columns = ['par_id', 'art_id', 'keyword', 'country', 'text', 'label']
df_raw['par_id'] = df_raw['par_id'].astype(str)

# Clean labels
df_raw = df_raw.dropna(subset=['text', 'label'])
df_raw['label'] = pd.to_numeric(df_raw['label'], errors='coerce')
df_raw = df_raw.dropna(subset=['label'])

df_raw['binary_label'] = df_raw['label'].apply(lambda x: 1 if x >= 2 else 0).astype(int)

# load splits
train_ids = pd.read_csv(base_path + 'train_semeval_parids-labels.csv')
dev_ids = pd.read_csv(base_path + 'dev_semeval_parids-labels.csv')
train_ids['par_id'] = train_ids['par_id'].astype(str)
dev_ids['par_id'] = dev_ids['par_id'].astype(str)

# merge
df_train = pd.merge(df_raw, train_ids[['par_id']], on='par_id', how='inner')
df_dev = pd.merge(df_raw, dev_ids[['par_id']], on='par_id', how='inner')

# test set
df_test = pd.read_csv(base_path + 'task4_test.tsv', sep='\t', header=None, on_bad_lines='skip')
df_test.columns = ['par_id', 'art_id', 'keyword', 'country', 'text']
df_test['text'] = df_test['text'].fillna("") # Prevent NaN errors

# df_train['text'] = df_train['keyword'] + " | " + df_train['country'].astype(str) + " : " + df_train['text']
# df_dev['text'] = df_dev['keyword'] + " | " + df_dev['country'].astype(str) + " : " + df_dev['text']
# df_test['text'] = df_test['keyword'] + " | " + df_test['country'].astype(str) + " : " + df_test['text']

print(f"Training samples: {len(df_train)}")
print(f"Dev samples: {len(df_dev)}")
print(f"Test samples: {len(df_test)}")

Loading datasets...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training samples: 8375
Dev samples: 2093
Test samples: 3832


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer, DebertaV2ForSequenceClassification, DebertaV2Tokenizer

# Implement Focal Loss for severe class imbalances
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha # Class weights

    def forward(self, inputs, targets):
        # Dynamically cast the weights to match the inputs' dtype and device!
        if self.alpha is not None:
            current_alpha = self.alpha.to(device=inputs.device, dtype=inputs.dtype)
        else:
            current_alpha = None

        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=current_alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Subclass HuggingFace Trainer to use our Focal Loss
class FocalTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = FocalLoss(alpha=self.class_weights, gamma=2.0)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

In [4]:
import torch
import torch.nn as nn
from transformers import Trainer, DebertaV2ForSequenceClassification, DebertaV2Tokenizer

# Subclass HuggingFace Trainer to use stable Class Weights instead of Focal Loss
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Safely cast weights to match the device and datatype
        if self.class_weights is not None:
            weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        else:
            weights = None

        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

In [11]:
from sklearn.model_selection import StratifiedKFold
from transformers import TrainingArguments, RobertaForSequenceClassification, RobertaTokenizer
import gc
import torch

# Hyperparameters
MODEL_NAME = 'roberta-base'
N_SPLITS = 5
MAX_LEN = 128
EPOCHS = 3

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# Prepare Dev and Test datasets
dev_encodings = tokenizer(df_dev['text'].tolist(), truncation=True, padding=True, max_length=MAX_LEN)
dev_dataset = SimpleDataset(dev_encodings)

test_encodings = tokenizer(df_test['text'].tolist(), truncation=True, padding=True, max_length=MAX_LEN)
test_dataset = SimpleDataset(test_encodings)

# Calculate class weights for WeightedTrainer
class_counts = df_train['binary_label'].value_counts().sort_index().values
weights = len(df_train) / (2.0 * class_counts)
weights_tensor = torch.tensor(weights, dtype=torch.float)

# Arrays to store predictions
oof_probs = np.zeros(len(df_train))
dev_probs_ensemble = np.zeros(len(df_dev))
test_probs_ensemble = np.zeros(len(df_test))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(df_train['text'], df_train['binary_label'])):
    print(f"\n========== FOLD {fold + 1}/{N_SPLITS} ==========")

    # Split data
    train_texts, val_texts = df_train['text'].iloc[train_idx].tolist(), df_train['text'].iloc[val_idx].tolist()
    train_labels, val_labels = df_train['binary_label'].iloc[train_idx].tolist(), df_train['binary_label'].iloc[val_idx].tolist()

    # Tokenize
    train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LEN)
    val_enc = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LEN)

    train_ds = SimpleDataset(train_enc, train_labels)
    val_ds = SimpleDataset(val_enc, val_labels)

    # Load fresh RoBERTa model
    model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f'./results_fold_{fold}',
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=32,
        warmup_steps=100,
        learning_rate=2e-5,
        # lr_scheduler_type="cosine",
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        fp16=True,
        report_to="none"
    )

    trainer = WeightedTrainer(
        class_weights=weights_tensor,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds
    )

    # Train
    trainer.train()

    # 1. Predict on Validation Fold (OOF)
    print("Predicting on Validation Fold...")
    val_preds = trainer.predict(val_ds).predictions
    val_probs = torch.nn.functional.softmax(torch.tensor(val_preds), dim=-1)[:, 1].numpy()
    oof_probs[val_idx] = val_probs

    # 2. Predict on Dev Set
    print("Predicting on Official Dev Set...")
    dev_preds = trainer.predict(dev_dataset).predictions
    dev_probs_ensemble += torch.nn.functional.softmax(torch.tensor(dev_preds), dim=-1)[:, 1].numpy() / N_SPLITS

    # 3. Predict on Test Set
    print("Predicting on Official Test Set...")
    test_preds = trainer.predict(test_dataset).predictions
    test_probs_ensemble += torch.nn.functional.softmax(torch.tensor(test_preds), dim=-1)[:, 1].numpy() / N_SPLITS

    # Cleanup memory before next fold
    del model, trainer, train_ds, val_ds
    torch.cuda.empty_cache()
    gc.collect()

print("\nEnsemble Training Complete!")



========== FOLD 1/5 ==========


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,0.407008
2,No log,0.380471
3,0.863495,0.437668


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Predicting on Validation Fold...


Predicting on Official Dev Set...


Predicting on Official Test Set...



Ensemble Training Complete!


In [12]:
from sklearn.metrics import precision_recall_curve, f1_score

# Find the optimal threshold using Out-Of-Fold predictions
train_labels_all = df_train['binary_label'].values
precisions, recalls, thresholds = precision_recall_curve(train_labels_all, oof_probs)

# Calculate F1 score for all thresholds
f1_scores = np.divide(2 * (precisions * recalls), (precisions + recalls),
                      out=np.zeros_like(precisions), where=(precisions + recalls) != 0)

optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal Threshold optimized for F1: {optimal_threshold:.4f}")
print(f"OOF F1-Score at optimal threshold: {f1_scores[optimal_idx]:.4f}")

# Apply the optimal threshold to Dev and Test ensemble probabilities
final_dev_preds = (dev_probs_ensemble >= optimal_threshold).astype(int)
final_test_preds = (test_probs_ensemble >= optimal_threshold).astype(int)

# Check Dev F1
dev_labels_all = df_dev['binary_label'].values
dev_f1 = f1_score(dev_labels_all, final_dev_preds)
print(f"\n>>> FINAL OFFICIAL DEV SET F1-SCORE: {dev_f1:.4f} <<<")
print("(Baseline was 0.48. If this is higher, you secured the points!)")

with open("dev.txt", "w") as f:
    for pred in final_dev_preds:
        f.write(f"{pred}\n")

with open("test.txt", "w") as f:
    for pred in final_test_preds:
        f.write(f"{pred}\n")

print("\nFiles 'dev.txt' and 'test.txt' have been generated successfully!")

Optimal Threshold optimized for F1: 0.7738
OOF F1-Score at optimal threshold: 0.5335

>>> FINAL OFFICIAL DEV SET F1-SCORE: 0.5857 <<<
(Baseline was 0.48. If this is higher, you secured the points!)

Files 'dev.txt' and 'test.txt' have been generated successfully!
